# TRL SFT Training Notebook (OpenEnv Submission)

This notebook runs a minimal Hugging Face TRL SFT training pass and exports metrics for submission evidence.

In [ ]:
!pip install -q trl transformers datasets accelerate torch

In [ ]:
from pathlib import Path
import json
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

BASE_MODEL = "distilgpt2"
OUTPUT_DIR = Path("/content/trl_sft_checkpoint")
METRICS_PATH = Path("/content/trl_sft_training_metrics.json")

In [ ]:
examples = [
    {"text": "### Instruction:\nDataset has missing values in email and phone columns.\n\n### Response:\n{\"action_type\": \"analyze\", \"target_columns\": [\"email\", \"phone\"], \"parameters\": {}, \"reasoning\": \"Inspect high-missing columns first.\"}"},
    {"text": "### Instruction:\nInvoice records contain duplicate invoice_id rows.\n\n### Response:\n{\"action_type\": \"deduplicate\", \"target_columns\": [\"invoice_id\"], \"parameters\": {\"subset\": [\"invoice_id\"], \"keep\": \"first\"}, \"reasoning\": \"Remove duplicate invoices by key.\"}"},
    {"text": "### Instruction:\nValidate numeric CSAT score range after cleaning.\n\n### Response:\n{\"action_type\": \"validate\", \"target_columns\": [\"csat_score\"], \"parameters\": {\"csat_score_type\": \"numeric\", \"csat_score_min\": 1, \"csat_score_max\": 5}, \"reasoning\": \"Ensure score range constraints.\"}"},
]
dataset = Dataset.from_list(examples)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
model.config.pad_token_id = tokenizer.pad_token_id

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_steps=1,
    save_strategy="no",
    report_to=[],
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args,
)

result = trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

metrics = {
    "base_model": BASE_MODEL,
    "train_examples": len(dataset),
    "train_runtime": result.metrics.get("train_runtime"),
    "train_steps_per_second": result.metrics.get("train_steps_per_second"),
    "train_loss": result.metrics.get("train_loss"),
    "global_step": result.metrics.get("global_step"),
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
metrics